# Monte Carlo Tournament Simulator

Uses the winning match model from `05_ensemble` (50/50 blend of the 6-learner Poisson ensemble + Dixon-Coles) to sample entire simulated World Cups ~50k times, producing tournament-level probabilities.

**Two forecasting paths:**
- **Path 1** — pre-tournament: simulate the whole thing from the draw.
- **Path 2** — live: condition on played matches, only simulate what's left.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from itertools import combinations

from src.evaluation import ranked_probability_score, rps_from_lambdas

## Setup

Reproduce the split from earlier notebooks. All 6 tuned learners + DC deployed (refit on train + eval) are loaded from `models/`.

In [2]:
# Load the cleaned dataset
results = pd.read_parquet('../data/interim/results_clean.parquet')

# Reproduce the train/eval/WC2026 split (same as previous notebooks)
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)
train_df = model_df[model_df['date'] < '2024-01-01']
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]
wc2026_df = model_df[wc2026_mask]

print(f'Train: {len(train_df)} | Eval: {len(eval_df)} | WC2026: {len(wc2026_df)}')

Train: 38900 | Eval: 2543 | WC2026: 79


In [3]:
# Load all deployed models from 05_ensemble (trained on train + eval set & tuned)
model_dir = Path('../models')
cb_tuned = joblib.load(model_dir / 'cb_deployed.joblib')
lgb_tuned = joblib.load(model_dir / 'lgb_deployed.joblib')
xgb_tuned = joblib.load(model_dir / 'xgb_deployed.joblib')
hgb_tuned = joblib.load(model_dir / 'hgb_deployed.joblib')
rf_tuned = joblib.load(model_dir / 'rf_deployed.joblib')
glm_tuned = joblib.load(model_dir / 'glm_deployed.joblib')

print('6 tuned learners loaded from disk')

# Load the DC model from 03 notebook
dc_deployed = joblib.load(model_dir / 'dc_deployed.joblib')
print('DC model deployed')

# Load feature-engineered model_df from 05_ensemble 
model_df = pd.read_parquet('../data/interim/model_df_featured.parquet')
print('model_df loaded from disk')

6 tuned learners loaded from disk
DC model deployed
model_df loaded from disk


## Team state snapshot

The simulator scores **hypothetical** fixtures (Argentina vs France in the final — no such row exists in `model_df`). We build the feature row on the fly from frozen state dicts.

`build_team_state` walks pre-WC matches chronologically once, storing each team's end-of-history Elo, rolling form, and last match date.

In [4]:
def build_team_state(matches, k=30, base_rating=1500, form_window=5):
    """Walk matches chronologically and snapshot each team's state at the end.

    Returns dict: {team: {'elo', 'recent_scored', 'recent_conceded', 'last_date'}}.
    Used to score hypothetical 2026 fixtures where feature rows don't yet exist.
    """
    matches = matches.sort_values('date').reset_index(drop=True)

    ratings = {}
    recent_scored = {}
    recent_conceded = {}
    last_date = {}

    for _, row in matches.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']
        home_goals = row['home_score']
        away_goals = row['away_score']

        # Elo update (same logic as add_elo_features)
        elo_home = ratings.get(home_team, base_rating)
        elo_away = ratings.get(away_team, base_rating)

        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))

        if home_goals > away_goals:
            actual_home = 1
        elif home_goals == away_goals:
            actual_home = 0.5
        else:
            actual_home = 0

        change = k * (actual_home - expected_home)
        ratings[home_team] = elo_home + change
        ratings[away_team] = elo_away - change

        # Rolling form - keep the last 'form_window' matches per team
        recent_scored.setdefault(home_team, []).append(home_goals)
        recent_conceded.setdefault(home_team, []).append(away_goals)
        recent_scored.setdefault(away_team, []).append(away_goals)
        recent_conceded.setdefault(away_team, []).append(home_goals)

        recent_scored[home_team] = recent_scored[home_team][-form_window:]
        recent_conceded[home_team] = recent_conceded[home_team][-form_window:]
        recent_scored[away_team] = recent_scored[away_team][-form_window:]
        recent_conceded[away_team] = recent_conceded[away_team][-form_window:]

        last_date[home_team] = row['date']
        last_date[away_team] = row['date']

    return {
        team: {
            'elo': ratings[team],
            'recent_scored': recent_scored[team],
            'recent_conceded': recent_conceded[team],
            'last_date': last_date[team]
        }
        for team in ratings 
    }

pre_wc_df = model_df[~wc2026_mask]
team_state = build_team_state(pre_wc_df)

# Sanity check: top 10 by elo 
top_teams = sorted(team_state.items(), key=lambda kv: kv[1]['elo'], reverse=True)[:10]
for team, state in top_teams:
    print(f"{team:20s} Elo {state['elo']:7.1f}  last {state['last_date'].date()}")

Argentina            Elo  2042.8  last 2026-06-09
Spain                Elo  2038.9  last 2026-06-08
France               Elo  1981.5  last 2026-06-08
Portugal             Elo  1948.1  last 2026-06-10
Brazil               Elo  1944.0  last 2026-06-06
England              Elo  1938.0  last 2026-06-10
Germany              Elo  1937.1  last 2026-06-06
Colombia             Elo  1926.5  last 2026-06-07
Netherlands          Elo  1911.1  last 2026-06-08
Japan                Elo  1908.6  last 2026-05-31


## H2H snapshot

Same walk, keyed by pair — running match count + cumulative goal-diff sum (from the alphabetically-first team's perspective, matching `add_h2h_features` in `05_ensemble`).

In [5]:
def build_h2h_state(matches):
    """Walk matches chronologically and snapshot H2H state at the end.

    Returns two dicts keyed by tuple(sorted([team_a, team_b])):
      - h2h_matches_played: count of prior meetings
      - h2h_gd_sum: cumulative goal difference from the alphabetically-first
                    team's perspective (matches add_h2h_features' convention)
    """
    matches = matches.sort_values('date').reset_index(drop=True)

    h2h_matches_played = {}
    h2h_gd_sum = {}

    for _, row in matches.iterrows():
        key = tuple(sorted([row['home_team'], row['away_team']]))
        reference_team = key[0]

        match_gd = row['home_score'] - row['away_score']
        if row['home_team'] != reference_team:
            match_gd = -match_gd

        h2h_matches_played[key] = h2h_matches_played.get(key, 0) + 1
        h2h_gd_sum[key] = h2h_gd_sum.get(key, 0) + match_gd 

    return h2h_matches_played, h2h_gd_sum

h2h_played, h2h_gd = build_h2h_state(pre_wc_df)

# Sanity check: Argentina vs Brazil should be a big number, both directions equal
key = tuple(sorted(['Argentina', 'Brazil']))
print(f"Argentina-Brazil: {h2h_played[key]} matches, cumulative GD {h2h_gd[key]:+d} (from {key[0]}'s perspective)")

key = tuple(sorted(['Germany', 'Italy']))
print(f"Germany-Italy:    {h2h_played[key]} matches, cumulative GD {h2h_gd[key]:+d} (from {key[0]}'s perspective)")

Argentina-Brazil: 60 matches, cumulative GD -24 (from Argentina's perspective)
Germany-Italy:    26 matches, cumulative GD +2 (from Germany's perspective)


## Fixture λ predictor

Given a hypothetical fixture, return `(λ_home, λ_away)`. Builds the same long-format 2-row frame the models were trained on (home-attacker + away-attacker), averages the 6 boosters, blends 50/50 with DC. Stateless — takes the state dicts as arguments.

In [6]:
ensemble_models = [cb_tuned, lgb_tuned, xgb_tuned, hgb_tuned, rf_tuned, glm_tuned]

ensemble_features = [
    'attacker_elo', 'defender_elo',
    'attacker_form_scored', 'attacker_form_conceded',
    'defender_form_scored', 'defender_form_conceded',
    'attacker_rest_days', 'defender_rest_days',
    'tournament_tier', 'neutral',
    'h2h_matches_played', 'h2h_goal_diff', 'is_home',
]

def predict_blend_lambdas(home_team, away_team, neutral, tournament_tier, fixture_date,
                          team_state, h2h_played, h2h_gd, ensemble_models, dc_model, blend_weight=0.5):
    """Return blended (λ_home, λ_away) for a single fixture.

    Builds the same long-format feature rows the models were trained on:
    two rows (home-attacker, away-attacker), predicts λ per side from each
    of the 6 boosters, averages them, then blends 50/50 with Dixon-Coles.
    """
    home_state = team_state[home_team]
    away_state = team_state[away_team]

    # Derived features
    home_rest = (fixture_date - home_state['last_date']).days
    away_rest = (fixture_date - away_state['last_date']).days
    home_form_scored = np.mean(home_state['recent_scored'])
    home_form_conceded = np.mean(home_state['recent_conceded'])
    away_form_scored = np.mean(away_state['recent_scored'])
    away_form_conceded = np.mean(away_state['recent_conceded'])

    key = tuple(sorted([home_team, away_team]))
    played = h2h_played.get(key, 0)
    if played > 0:
        avg_gd_reference = h2h_gd[key] / played
        h2h_diff_home_view = avg_gd_reference if home_team == key[0] else -avg_gd_reference
    else:
        h2h_diff_home_view = np.nan
    
    is_home_flag = 0 if neutral else 1

    home_attacker_row = {
        'attacker_elo': home_state['elo'],
        'defender_elo': away_state['elo'],
        'attacker_form_scored': home_form_scored,
        'attacker_form_conceded': home_form_conceded,
        'defender_form_scored': away_form_scored,
        'defender_form_conceded': away_form_conceded,
        'attacker_rest_days': home_rest,
        'defender_rest_days': away_rest,
        'tournament_tier': tournament_tier,
        'neutral': neutral,
        'h2h_matches_played': played,
        'h2h_goal_diff': h2h_diff_home_view,
        'is_home': is_home_flag,
    }

    away_attacker_row = {
        **home_attacker_row,
        'attacker_elo': away_state['elo'],
        'defender_elo': home_state['elo'],
        'attacker_form_scored': away_form_scored,
        'attacker_form_conceded': away_form_conceded,
        'defender_form_scored': home_form_scored,
        'defender_form_conceded': home_form_conceded,
        'attacker_rest_days': away_rest,
        'defender_rest_days': home_rest,
        'h2h_goal_diff': -h2h_diff_home_view,   # sign-flip for the away attacker
        'is_home': 0,                            # away side never gets home advantage
    }

    X_ensemble = pd.DataFrame([home_attacker_row, away_attacker_row], columns=ensemble_features)
    ensemble_preds = np.mean([m.predict(X_ensemble) for m in ensemble_models], axis=0)
    lambda_home_ensemble, lambda_away_ensemble = ensemble_preds

    # DC uses its own schema — (attacking_team, defending_team, is_home)
    dc_rows = pd.DataFrame({
        'attacking_team': [home_team, away_team],
        'defending_team': [away_team, home_team],
        'is_home': [is_home_flag, 0],
    })

    lambda_home_dc, lambda_away_dc = dc_model.predict(dc_rows)

    lambda_home = blend_weight * lambda_home_ensemble + (1 - blend_weight) * lambda_home_dc
    lambda_away = blend_weight * lambda_away_ensemble + (1 - blend_weight) * lambda_away_dc

    return lambda_home, lambda_away

# Sanity check: Argentina vs Brazil at a neutral WC venue
lam_home, lam_away = predict_blend_lambdas(
    'Argentina', 'Brazil',
    neutral=True, tournament_tier=3,
    fixture_date=pd.Timestamp('2026-07-15'),
    team_state=team_state, h2h_played=h2h_played, h2h_gd=h2h_gd,
    ensemble_models=ensemble_models, dc_model=dc_deployed,
)
print(f"λ Argentina {lam_home:.2f} | Brazil {lam_away:.2f}")

λ Argentina 1.07 | Brazil 0.95


## Group draw

Reconstruct the 12 groups from `wc2026_df`'s first 72 matches (each team's group = itself + its 3 group-stage opponents). Relabel to official FIFA letters via anchor teams — makes the bracket wiring deterministic across kernel restarts (set-iteration order is hash-randomized).

In [7]:
group_stage_df = wc2026_df.sort_values('date').head(72)

# For each team, collect the set of opponents they faced in group stage
opponents = {}
for _, row in group_stage_df.iterrows():
    opponents.setdefault(row['home_team'], set()).add(row['away_team'])
    opponents.setdefault(row['away_team'], set()).add(row['home_team'])

# Each team's group = themselves + their 3 opponents. Dedupe by frozenset.
groups = {frozenset({team} | opps) for team, opps in opponents.items()}
groups = [sorted(g) for g in groups]

print(f"{len(groups)} groups")

# One anchor team per official group (any unique team in each group works)
official_anchors = {
    'A': 'Mexico',
    'B': 'Canada',
    'C': 'Brazil',
    'D': 'United States',
    'E': 'Germany',
    'F': 'Netherlands',
    'G': 'Belgium',
    'H': 'Spain',
    'I': 'France',
    'J': 'Argentina',
    'K': 'Portugal',
    'L': 'England',
}

# For each derived group, find which anchor it contains → that's its official letter
groups = {
    letter: next(g for g in groups if anchor in g)
    for letter, anchor in official_anchors.items()
}

for letter in sorted(groups):
    print(f"Group {letter}: {', '.join(groups[letter])}")

12 groups
Group A: Czech Republic, Mexico, South Africa, South Korea
Group B: Bosnia and Herzegovina, Canada, Qatar, Switzerland
Group C: Brazil, Haiti, Morocco, Scotland
Group D: Australia, Paraguay, Turkey, United States
Group E: Curaçao, Ecuador, Germany, Ivory Coast
Group F: Japan, Netherlands, Sweden, Tunisia
Group G: Belgium, Egypt, Iran, New Zealand
Group H: Cape Verde, Saudi Arabia, Spain, Uruguay
Group I: France, Iraq, Norway, Senegal
Group J: Algeria, Argentina, Austria, Jordan
Group K: Colombia, DR Congo, Portugal, Uzbekistan
Group L: Croatia, England, Ghana, Panama


## λ cache — group pairings

Precompute λ's for the 72 intra-group pairings once. Simulator inner loop is dict lookups instead of ~3.6M model calls (72 fixtures × 50k iterations).

In [8]:
fixture_date = pd.Timestamp('2026-07-15')

lambda_cache = {}
for letter, teams in groups.items():
    for team_a, team_b in combinations(teams, 2):
        lam_a, lam_b = predict_blend_lambdas(
            team_a, team_b,
            neutral=True, tournament_tier=3, fixture_date=fixture_date,
            team_state=team_state, h2h_played=h2h_played, h2h_gd=h2h_gd,
            ensemble_models=ensemble_models, dc_model=dc_deployed
        )
        lambda_cache[frozenset({team_a, team_b})] = {team_a: lam_a, team_b: lam_b}

print(f'Cached λ for {len(lambda_cache)} group pairings')   # should print 72

Cached λ for 72 group pairings


## Group-phase simulator

- `simulate_group` — play 6 matches, standings sorted by pts / GD / GF (FIFA tiebreak).
- `simulate_group_phase` — run all 12 groups → top-2 direct + 8 best 3rd-placed → 32 R32 qualifiers.

In [9]:
def simulate_group(teams, lambda_cache, rng):
    """Play the 6 group matches; return standings sorted by pts → GD → goals scored.

    Returns list of length 4, best first: [{team, pts, gd, gf}, ...].
    """
    standings = {t: {'team': t, 'pts': 0, 'gd': 0, 'gf': 0} for t in teams}

    for team_a, team_b in combinations(teams, 2):
        cache = lambda_cache[frozenset({team_a, team_b})]
        goals_a = rng.poisson(cache[team_a])
        goals_b = rng.poisson(cache[team_b])

        if goals_a > goals_b:
            standings[team_a]['pts'] += 3
        elif goals_a == goals_b:
            standings[team_a]['pts'] += 1
            standings[team_b]['pts'] += 1
        else:
            standings[team_b]['pts'] += 3
        
        standings[team_a]['gd'] += goals_a - goals_b
        standings[team_b]['gd'] += goals_b - goals_a
        standings[team_a]['gf'] += goals_a
        standings[team_b]['gf'] += goals_b

    return sorted(standings.values(),
                  key=lambda s: (s['pts'], s['gd'], s['gf']),
                  reverse=True)

In [10]:
def simulate_group_phase(groups, lambda_cache, rng):
    """Simulate all 12 groups; return top-2 per group + 8 best 3rd-placed teams."""
    group_results = {}
    thirds = []

    for letter, teams in groups.items():
        ranked = simulate_group(teams, lambda_cache, rng)
        group_results[letter] = {
            'top1': ranked[0]['team'],
            'top2': ranked[1]['team'],
            'third_standing': ranked[2]
        }

        thirds.append({**ranked[2], 'group': letter})

    thirds_ranked = sorted(thirds, key=lambda s: (s['pts'], s['gd'], s['gf']), reverse=True)
    qualified_thirds = [t['team'] for t in thirds_ranked[:8]]

    return group_results, qualified_thirds

rng = np.random.default_rng(seed=42)
group_results, qualified_thirds = simulate_group_phase(groups, lambda_cache, rng)

for letter in sorted(groups):
    r = group_results[letter]
    third = r['third_standing']
    print(f"Group {letter}: 1st {r['top1']:<20s} 2nd {r['top2']:<20s} 3rd {third['team']:<20s} ({third['pts']}p, GD {third['gd']:+d})")

print(f"\n8 qualified 3rd-placed teams: {qualified_thirds}")

Group A: 1st Czech Republic       2nd Mexico               3rd South Africa         (6p, GD +0)
Group B: 1st Switzerland          2nd Canada               3rd Bosnia and Herzegovina (4p, GD +0)
Group C: 1st Morocco              2nd Brazil               3rd Scotland             (4p, GD -2)
Group D: 1st Turkey               2nd United States        3rd Australia            (3p, GD -2)
Group E: 1st Ivory Coast          2nd Ecuador              3rd Germany              (4p, GD +3)
Group F: 1st Sweden               2nd Japan                3rd Netherlands          (4p, GD +0)
Group G: 1st Belgium              2nd Egypt                3rd New Zealand          (4p, GD -2)
Group H: 1st Spain                2nd Uruguay              3rd Cape Verde           (3p, GD -4)
Group I: 1st France               2nd Norway               3rd Senegal              (2p, GD -2)
Group J: 1st Algeria              2nd Argentina            3rd Austria              (4p, GD -1)
Group K: 1st Portugal             2nd 

## λ cache — extend to all pairings

Knockouts pair teams **across** groups (Argentina vs Spain could meet in a SF), so we need λ's for every possible matchup. Expand from 72 to all `C(48, 2) = 1128` pairings.

In [11]:
all_teams = sorted({t for teams in groups.values() for t in teams})

for team_a, team_b in combinations(all_teams, 2):
    key = frozenset({team_a, team_b})
    if key in lambda_cache:
        continue
    
    lam_a, lam_b = predict_blend_lambdas(
        team_a, team_b,
        neutral=True, tournament_tier=3, fixture_date=fixture_date,
        team_state=team_state, h2h_played=h2h_played, h2h_gd=h2h_gd,
        ensemble_models=ensemble_models, dc_model=dc_deployed
    )

    lambda_cache[key] = {team_a: lam_a, team_b: lam_b}

print(f'λ cache now covers {len(lambda_cache)} pairings') # should print 1128 = C(48, 2)

λ cache now covers 1128 pairings


## Knockout simulator

- `simulate_knockout_match` — one match; on draw, resolve with a 50/50 shootout.
- `simulate_bracket` — random-pair the 32 qualifiers (v1 punt on the FIFA R32 lookup) and run R32 → R16 → QF → SF → Final.

In [12]:
def simulate_knockout_match(team_a, team_b, lambda_cache, rng):
    """One knockout match. On draw, resolve with a 50/50 shootout.

    Returns the winning team's name.
    """
    cache = lambda_cache[frozenset({team_a, team_b})]
    goals_a = rng.poisson(cache[team_a])
    goals_b = rng.poisson(cache[team_b])

    if goals_a > goals_b:
        return team_a
    if goals_b > goals_a:
        return team_b
    return rng.choice([team_a, team_b]) # penalty shootout, ~50/50

In [13]:
def simulate_bracket(qualifiers, lambda_cache, rng):
    """Random-pair the 32 qualifiers, then run R32 → R16 → QF → SF → Final.

    Returns dict {round: [winners]} plus the champion.
    """
    round_names = ['R32', 'R16', 'QF', 'SF', 'Final']
    bracket = list(qualifiers)
    rng.shuffle(bracket)
    results = {}

    for round_name in round_names:
        winners = []
        for i in range(0, len(bracket), 2):
            winner = simulate_knockout_match(bracket[i], bracket[i + 1], lambda_cache, rng)
            winners.append(winner)
        
        results[round_name] = winners
        bracket = winners
    
    champion = bracket[0]
    return results, champion

## One full tournament

Stitch group phase + bracket together. Returns the full trail (`group_results`, `knockout_results`, `champion`) for one simulated World Cup.

In [14]:
def simulate_tournament(groups, lambda_cache, rng):
    """One full simulated WC: group phase → 32 qualifiers → knockouts → champion."""
    group_results, qualified_thirds = simulate_group_phase(groups, lambda_cache, rng)

    qualifiers = []
    for r in group_results.values():
        qualifiers.extend([r['top1'], r['top2']])
    qualifiers.extend(qualified_thirds)

    knockout_results, champion = simulate_bracket(qualifiers, lambda_cache, rng)
    return group_results, knockout_results, champion

# Sanity check - one full tournament
rng = np.random.default_rng(seed=42)
group_results, knockout_results, champion = simulate_tournament(groups, lambda_cache, rng)

print(f"Champion: {champion}")
print(f"Finalists: {knockout_results['SF']}")
print(f"Semifinalists: {knockout_results['QF']}")

Champion: Netherlands
Finalists: ['France', 'Netherlands']
Semifinalists: ['Argentina', 'France', 'England', 'Netherlands']


## Path 1 — Pre-tournament forecast

50k iterations from the draw. Champion probabilities across all 48 teams.

In [15]:
rng = np.random.default_rng(seed=42)
champions = []
for _ in range(50_000):
    _, _, champion = simulate_tournament(groups, lambda_cache, rng)
    champions.append(champion)

pd.Series(champions).value_counts(normalize=True).head(15)

Argentina      0.12016
Spain          0.11986
Brazil         0.09228
France         0.07734
England        0.07234
Portugal       0.06702
Germany        0.05760
Netherlands    0.04932
Colombia       0.04638
Belgium        0.04058
Morocco        0.03058
Ecuador        0.02382
Croatia        0.02342
Japan          0.02196
Uruguay        0.02028
Name: proportion, dtype: float64

## Path 2 — Live conditioning

Fix played WC matches as facts, only simulate the remainder. Distribution collapses to teams still alive.

In [16]:
# Path 2 — Live-conditioning sim, updated 2026-07-12
# Fixed facts: France beat Morocco, Spain beat Belgium, England beat Norway, Argentina beat Switzerland.
# Remaining: 2 SFs + Final = 3 matches per iteration.

rng = np.random.default_rng(seed=42)
champions_live = []

for _ in range(50_000):
    sf1_winner = simulate_knockout_match('France', 'Spain', lambda_cache, rng)
    sf2_winner = simulate_knockout_match('England', 'Argentina', lambda_cache, rng)
    champion = simulate_knockout_match(sf1_winner, sf2_winner, lambda_cache, rng)
    champions_live.append(champion)

pd.Series(champions_live).value_counts(normalize=True)

Argentina    0.29060
Spain        0.28884
France       0.21334
England      0.20722
Name: proportion, dtype: float64